# Causal EEW v3: source-preserving repairs and pilot gate

The v2 noise scale uses future innovations through whole-record SD. Its gamma templates use future envelope maxima. True future-only perturbations change tail values without changing earlier innovations/envelopes or shared magnitude/distance/decay parameters. Changing shared physical parameters is not an invariance test.

Reuse the saved v2 earthquake class's velocities, Q attenuation, magnitude scaling, carriers and gamma time constants. Replace realized global noise SD with fixed filter impulse-response energy; replace sampled envelope maxima with the analytic gamma maximum at tau=2*tau_d. Separate arrival/background/profile RNG streams. Pink-like backgrounds require a causal FIR approximation instead of record-length-dependent FFT generation; transient envelopes start at their onset rather than having Gaussian pre-onset tails. No real arrival time is moved. Stored S index is the first grid sample at/after its continuous arrival (ceil); eligibility uses strict integer cutoff < S index. P windows remain oracle-aligned.


In [1]:
from pathlib import Path
import sys,ast,json,hashlib,copy,math
sys.dont_write_bytecode=True
from typing import Dict,Any,List,Tuple
import numpy as np
import pandas as pd
import h5py
from scipy.signal import butter,sosfilt,lfilter
ROOT=Path(r"C:/Users/choud/Desktop/Minor project/cnn/cnn wave")
V2=ROOT/"eew_causal_v2";V3=ROOT/"eew_causal_v3"
for d in [V3,V3/"dataset",V3/"reports",V3/"configs"]:d.mkdir(parents=True,exist_ok=True)
def readj(p):return json.loads(p.read_text(encoding="utf-8"))
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""):h.update(b)
    return h.hexdigest()
def plain(v):
    if isinstance(v,dict):return {str(k):plain(x) for k,x in v.items()}
    if isinstance(v,(list,tuple,np.ndarray)):return [plain(x) for x in v]
    if isinstance(v,np.generic):return plain(v.item())
    return v
def savej(p,v):
    t=p.with_suffix(p.suffix+".tmp");t.write_text(json.dumps(plain(v),indent=2,allow_nan=False),encoding="utf-8");t.replace(p)
prior=readj(V2/"codex_independent_audit/audit_results.json")
protected={ROOT/rel for rel in prior["protected_after"]}
protected.update(p for p in (V2/"codex_independent_audit").iterdir() if p.is_file())
before={p.relative_to(ROOT).as_posix():sha(p) for p in sorted(protected)}
assert all(before[k]==v for k,v in prior["protected_after"].items())
if (V3/"configs/protected_sources_before.json").exists():
    assert readj(V3/"configs/protected_sources_before.json")==before
else:savej(V3/"configs/protected_sources_before.json",before)
params=readj(V2/"configs/causal_simulation_parameters.json")
nb2=readj(V2/"02_generate_causal_eew_dataset.ipynb")
v2source="".join(nb2["cells"][2]["source"])
v2tree=ast.parse(v2source)
v2classes=ast.unparse(ast.Module(body=[n for n in v2tree.body if isinstance(n,ast.ClassDef)],type_ignores=[]))
exec(compile(v2classes,"protected_v2_class_definitions_only","exec"))
fs=100.;DT=.01
# Filter gains depend only on a fixed impulse and coefficients, never a waveform's realized values.
sos=butter(4,[.2,25.],btype="bandpass",fs=100,output="sos")
pink_fir=np.ones(128)
for k in range(1,len(pink_fir)):pink_fir[k]=pink_fir[k-1]*(k-.5)/k
impulse=np.zeros(32768);impulse[0]=1
impulse_band=sosfilt(sos,impulse)
impulse_pink=sosfilt(sos,lfilter(pink_fir,[1.],impulse))
GAINS={"band":float(np.linalg.norm(impulse_band)),"pink":float(np.linalg.norm(impulse_pink))}
assert abs(np.linalg.norm(impulse_band[:16384])-GAINS["band"])<1e-12
assert abs(np.linalg.norm(impulse_pink[:16384])-GAINS["pink"])<1e-12
def stream(seed,domain,role):
    return np.random.default_rng(np.random.SeedSequence([int(seed),int(domain),int(role)]))
def gamma_peak(tau_r,tau_d):
    return (2*tau_d/tau_r)**2*np.exp(-2)
# V3 class comes from the actual v2 saved-data class, with explicit narrow transformations.
eq_node=next(n for n in v2tree.body if isinstance(n,ast.ClassDef) and n.name=="CausalEEWGenerator")
eq_source=ast.get_source_segment(v2source,eq_node)
replacements=[
 ("class CausalEEWGenerator:","class CausalEEWGeneratorV3:"),
 ("self.time_axis = np.linspace(0.0, (self.n_samples - 1) * self.dt, self.n_samples)",
  "self.time_axis = np.arange(self.n_samples, dtype=np.float64) * self.dt\n        self.interventions = {}"),
 ("s_sample = int(np.round(ts_sec * self.fs))","s_sample = int(np.ceil(ts_sec * self.fs))"),
 ("max_env_p = np.max(env_p) if len(env_p) > 0 and np.max(env_p) > 0 else 1.0","max_env_p = gamma_peak(tau_r_p, tau_d_p)"),
 ("max_env_s = np.max(env_s) if len(env_s) > 0 and np.max(env_s) > 0 else 1.0","max_env_s = gamma_peak(tau_r_s, tau_d_s)"),
 ("env_p = env_p / max_env_p","env_p = env_p / max_env_p\n        if 'p_coda' in self.interventions:\n            after, gain = self.interventions['p_coda']\n            env_p[tau_p >= after] *= gain"),
 ("env_s = env_s / max_env_s","env_s = env_s / max_env_s\n        if 's_coda' in self.interventions:\n            after, gain = self.interventions['s_coda']\n            env_s[tau_s >= after] *= gain"),
 ("raw_noise = rng.normal(0.0, 1.0, self.n_samples)",
  "raw_noise = stream(seed, 11, 1).normal(0.0, 1.0, self.n_samples)\n        if 'noise' in self.interventions:\n            after, gain = self.interventions['noise']\n            raw_noise[self.time_axis >= tp_sec + after] *= gain"),
 ("raw_std = np.std(filtered_noise)","raw_std = GAINS['band']"),
 ("pre_s_v1 = bool(ts_sec > tp_sec + 1.0)","pre_s_v1 = bool(s_sample > p_sample + 100)"),
 ("pre_s_v2 = bool(ts_sec > tp_sec + 2.0)","pre_s_v2 = bool(s_sample > p_sample + 200)"),
 ("pre_s_v3 = bool(ts_sec > tp_sec + 3.0)","pre_s_v3 = bool(s_sample > p_sample + 300)")]
for old,new in replacements:
    assert eq_source.count(old)==1,old
    eq_source=eq_source.replace(old,new)
eq_tree=ast.parse(eq_source)
method=next(n for n in eq_tree.body[0].body if isinstance(n,ast.FunctionDef) and n.name=="synthesize_earthquake_window")
ret=next(n for n in method.body if isinstance(n,ast.Return))
for k,expr in {"parent_raw":"full_waveform","background":"noise","P_component":"pw","S_component":"sw",
               "raw_innovations":"raw_noise","filtered_innovations":"filtered_noise","context_mean":"mean_pre",
               "context_std":"std_pre","reference_sample":"p_sample"}.items():
    ret.value.keys.append(ast.Constant(k));ret.value.values.append(ast.parse(expr,mode="eval").body)
ast.fix_missing_locations(eq_tree)
exec(compile(eq_tree,"v3_repaired_eq_class","exec"))
def normalize_windows(parent,reference):
    pre=parent[reference-100:reference]
    assert len(pre)==100
    mean=float(np.mean(pre));std=float(np.std(pre))+1e-8
    out={"parent_raw":parent,"reference_sample":reference,"context_mean":mean,"context_std":std}
    for t in [1,2,3]:
        raw=parent[reference-100:reference+t*100]
        assert raw.shape==(100+t*100,)
        out[f"w{t}_raw"]=raw.astype(np.float32)
        out[f"w{t}_norm"]=((raw-mean)/std).astype(np.float32)
        out[f"w{t}_post_raw"]=parent[reference:reference+t*100].copy()
    return out
class CausalNoiseWindowGeneratorV3:
    def __init__(self,p):
        self.p=p;self.n=int(round(p["duration_seconds"]*100));self.time=np.arange(self.n)*.01
        self.interventions={}
    def generate_noise_window(self,noise_type,seed,target_rms=.05):
        # Reference/profile parameters do not consume ambient-innovation RNG state.
        ref=int(round((3+4*stream(seed,22,0).uniform())*100))
        innovations=stream(seed,22,1).normal(size=self.n)
        if "noise" in self.interventions:
            after,gain=self.interventions["noise"];innovations[self.time>=ref*.01+after]*=gain
        if noise_type=="colored_pink":
            filtered=sosfilt(sos,lfilter(pink_fir,[1.],innovations));gain=GAINS["pink"]
        else:
            filtered=sosfilt(sos,innovations);gain=GAINS["band"]
        background=filtered*(target_rms/gain)
        if noise_type=="modulated_nonstationary":
            r=stream(seed,22,2);freq=r.uniform(.01,.04);phase=r.uniform(0,2*np.pi);depth=r.uniform(.25,.45)
            background*=1+depth*np.sin(2*np.pi*freq*self.time+phase)
        elif noise_type=="transient_disturbances":
            r=stream(seed,22,3)
            # Fixed acquisition-independent schedule; extending/truncating the record never resamples parameters.
            for _ in range(int(r.integers(1,4))):
                tc=r.uniform(0,100);duration=r.uniform(.5,3);amp=r.uniform(.5,2)*target_rms
                tau=self.time-tc;active=tau>=0
                env=np.zeros(self.n);u=tau[active]/duration
                env[active]=(u**2)*np.exp(2-u)/4
                background+=amp*env*np.sin(2*np.pi*3*tau)
        elif noise_type not in ["colored_pink","gaussian_bandlimited"]:
            raise ValueError(noise_type)
        parent=background.astype(np.float32)
        out=normalize_windows(parent,ref);out.update(background=parent.copy(),raw_innovations=innovations)
        return out
def magnitude_mae(pred,target,mask):
    # A mask-aware dataset usage contract, not a trained model.
    mask=np.asarray(mask,bool)
    return float(np.mean(np.abs(np.asarray(pred)[mask]-np.asarray(target)[mask]))) if mask.any() else None
configuration={"version":"causal_eew_v3","source_v2_notebook_sha256":sha(V2/"02_generate_causal_eew_dataset.ipynb"),
 "parameters":params,"sample_rate_hz":100,"horizons_s":[1,2,3],"pre_context_samples":100,
 "noise_gains":GAINS,"gain_calibration":"fixed32768-sample filter impulse; tail convergence checked at16384",
 "gamma_normalization":"analytic maximum at tau=2*tau_d; no sampled future maximum",
 "RNG":"P offset legacy default_rng(seed) one draw; earthquake noise SeedSequence(seed,11,1); noise reference/background/profile independent roles (seed,22,0/1/2/3)",
 "pink_change":"causal128-tap fractional-integration FIR; approximate colored pink-like spectrum, not exact FFT synthesis",
 "arrival_policy":"P sampled at exact prescribed grid onset; continuous S unchanged; S index=ceil(S_seconds*100), first sample at/after onset",
 "eligibility":"label==1 and reference_sample+100*tau < s_arrival_sample; noise pre_S false; classification_eligible=(label==0)|pre_S",
 "regression":"magnitude_mask=(label==1), magnitude_mask_tau=magnitude_mask & pre_S_tau; noise magnitude sentinel=-1",
 "scope":"synthetic oracle-aligned benchmark; no detected trigger or operational EEW claim",
 "physical_units":"synthetic relative amplitude, not calibrated instrument displacement"}
savej(V3/"configs/simulation_v3.json",configuration)
print("Root causes mapped; fixed gains",GAINS,"; original protected files",len(before))


Root causes mapped; fixed gains {'band': 0.7043400785612605, 'pink': 1.3103644084803956} ; original protected files 44


In [2]:
# Exact v2 failure reproduction using only the audit's instrument() definition.
audit_nb=readj(V2/"codex_independent_audit/dataset_audit.ipynb")
audit_source="\n".join("".join(c["source"]) for c in audit_nb["cells"] if c["cell_type"]=="code")
instrument_node=next(n for n in ast.parse(audit_source).body if isinstance(n,ast.FunctionDef) and n.name=="instrument")
class_source=v2classes
exec(compile(ast.Module(body=[instrument_node],type_ignores=[]),"v2_audit_instrument_definition","exec"))
meta=pd.read_csv(ROOT/"data wave/synthetic_waveform_simulator/output/phase2_travel_time/metadata_with_theoretical_arrivals.csv").set_index("source_id")
def maxdiff(a,b):return float(np.max(np.abs(np.asarray(a,dtype=float)-np.asarray(b,dtype=float))))
v2_reproduction=[];tests=[]
for case in prior["counterfactuals"]:
    row=meta.loc[case["source_id"]]
    kw=dict(magnitude=float(row.Magnitude_Value),hypocentral_distance_km=float(row.hypocentral_distance_km),
            theoretical_sp_sec=float(row.theoretical_sp_time_seconds),seed=int(case["seed"]),target_noise_rms=.05)
    old=instrument("original")(params).synthesize_earthquake_window(**kw)
    reproduced={}
    for mode in ["future_noise","future_P_coda"]:
        changed=instrument(mode)(params).synthesize_earthquake_window(**kw)
        got=maxdiff(old["w3_norm"],changed["w3_norm"])
        reproduced[mode]=got
        assert got==case[mode]["saved3s_normalized_difference"],"v2 evidence changed"
    v2_reproduction.append(dict(source_id=case["source_id"],seed=kw["seed"],**reproduced))
    base=CausalEEWGeneratorV3(params).synthesize_earthquake_window(**kw)
    p=base["p_sample"];s=base["s_sample"]
    rowtests={"source_id":case["source_id"],"seed":kw["seed"],"magnitude":kw["magnitude"],"SP":kw["theoretical_sp_sec"]}
    def compare(name,changed,stop):
        raw=maxdiff(base["parent_raw"][:stop],changed["parent_raw"][:stop])
        bn=(base["parent_raw"]-base["context_mean"])/base["context_std"]
        cn=(changed["parent_raw"]-changed["context_mean"])/changed["context_std"]
        norm=maxdiff(bn[:stop],cn[:stop])
        rowtests[name]={"raw":raw,"normalized":norm}
        assert raw==0.0 and norm==0.0,f"CRITICAL PILOT FAILURE {name}"
    gen=CausalEEWGeneratorV3(params);gen.sp_ratio*=4
    compare("A_S_amplitude_x4",gen.synthesize_earthquake_window(**kw),s)
    gen=CausalEEWGeneratorV3(params);gen.interventions={"s_coda":(5.,1.5)}
    compare("B_future_S_coda_plus50pct",gen.synthesize_earthquake_window(**kw),min(10000,int(np.ceil((base["ts_sec"]+5)*100))))
    gen=CausalEEWGeneratorV3(params);gen.interventions={"noise":(8.,4.)}
    compare("C_future_noise",gen.synthesize_earthquake_window(**kw),p+800)
    for gain in [1.5,4.]:
        gen=CausalEEWGeneratorV3(params);gen.interventions={"p_coda":(4.,gain)}
        compare("D_future_P_coda_x"+str(gain),gen.synthesize_earthquake_window(**kw),p+400)
    for duration in [12.,140.]:
        gen=CausalEEWGeneratorV3({**params,"duration_seconds":duration})
        changed=gen.synthesize_earthquake_window(**kw)
        compare("E_duration_"+str(duration),changed,min(len(base["parent_raw"]),len(changed["parent_raw"])))
    altered=base["parent_raw"].copy();altered[p+300:]+=17.
    after=normalize_windows(altered,p)
    d=max(maxdiff(base[f"w{t}_norm"],after[f"w{t}_norm"]) for t in [1,2,3])
    rowtests["F_future_after_preprocessing"]={"raw":0.,"normalized":d};assert d==0
    tests.append(rowtests)
noise_tests=[]
for typ in ["gaussian_bandlimited","colored_pink","modulated_nonstationary","transient_disturbances"]:
    for seed in [42,71,1234,9099]:
        base=CausalNoiseWindowGeneratorV3(params).generate_noise_window(typ,seed)
        ref=base["reference_sample"]
        errors={}
        for duration in [12.,140.]:
            other=CausalNoiseWindowGeneratorV3({**params,"duration_seconds":duration}).generate_noise_window(typ,seed)
            stop=min(len(base["parent_raw"]),len(other["parent_raw"]))
            errors["parent_prefix_"+str(duration)]=maxdiff(base["parent_raw"][:stop],other["parent_raw"][:stop])
            errors["normalized_"+str(duration)]=maxdiff(base["w3_norm"],other["w3_norm"])
        g=CausalNoiseWindowGeneratorV3(params);g.interventions={"noise":(8.,4.)}
        other=g.generate_noise_window(typ,seed)
        errors["future_noise_prefix"]=maxdiff(base["parent_raw"][:ref+800],other["parent_raw"][:ref+800])
        errors["future_noise_norm"]=maxdiff(base["w3_norm"],other["w3_norm"])
        assert max(errors.values())==0.0,"CRITICAL NOISE PILOT FAILURE"
        noise_tests.append(dict(profile=typ,seed=seed,errors=errors))
# Explicit sentinel exclusion demonstration: poison only noise labels; masked result stays identical.
target=np.array([3.,-1.,5.,-1.]);prediction=np.array([3.2,9.,4.8,100.]);mask=np.array([1,0,1,0],bool)
poison=target.copy();poison[~mask]=1e9
assert magnitude_mae(prediction,target,mask)==magnitude_mae(prediction,poison,mask)
assert magnitude_mae(prediction,target,np.zeros(4,bool)) is None
pilot={"status":"PASS","strict_tolerance":0.0,"earthquake_cases":tests,"noise_cases":noise_tests,
       "exact_v2_reproduction":v2_reproduction,"v2_future_noise_max":max(r["future_noise"] for r in v2_reproduction),
       "v2_future_P_coda_max":max(r["future_P_coda"] for r in v2_reproduction),
       "v3_max_prefix_difference":0.0,"masked_regression_contract":"PASS",
       "shared_parameter_policy":"Magnitude, distance, decay parameters and past innovations held fixed; only future-tail values changed. Record duration is not a physical source parameter.",
       "simulator_definition_sha256":hashlib.sha256(ast.unparse(eq_tree).encode()).hexdigest()}
savej(V3/"reports/v3_counterfactual_results.json",pilot)
savej(V3/"configs/pilot_gate.json",{"status":"PASS","pilot_sha256":sha(V3/"reports/v3_counterfactual_results.json"),
     "configuration_sha256":sha(V3/"configs/simulation_v3.json"),"strict_tolerance":0.0})
assert all(sha(ROOT/rel)==value for rel,value in before.items())
print("Pilot PASS:15 earthquake cases,16 noise cases; all A-F maxima exactly0. V2 failures reproduced:",
      pilot["v2_future_noise_max"],pilot["v2_future_P_coda_max"])


Pilot PASS:15 earthquake cases,16 noise cases; all A-F maxima exactly0. V2 failures reproduced: 361.7903289794922 99.57305908203125
